In [1]:
# ! pip install selenium
# ! pip install selenium-stealth
# ! pip install beautifulsoup4
# ! pip install lxml
# ! pip install requests
# ! pip install pandas

In [1]:
from selenium import webdriver
from selenium_stealth import stealth

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import requests
import csv
import time
from bs4 import BeautifulSoup

In [3]:
# Set up Chrome options
chrome_options = Options()
# chrome_options.add_argument("--headless")  # Run headless Chrome
chrome_options.add_argument("--window-size=1920,1080")  # Set window size
# chrome_options.add_argument('--headless=new') # ensure GUI is off
chrome_options.add_argument('start-maximized') #sets the browser to maximixed view
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-search-engine-choice-screen")
chrome_options.add_argument('disable-infobars')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--window-size=2560,1440") # set specific window size for the browser,
chrome_options.add_argument("--incognito") # set the browser mode to incognito
chrome_options.add_argument('--enable-javascript')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # selenium stealth option to enable automation
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument("lang=en")
chrome_options.add_argument("--disable-extensions")
driver = webdriver.Chrome(options=chrome_options)

#if you did not set the chrome driver to you environmental variable, uncomment the line below.
# driver = webdriver.Chrome(options=chrome_options, executable_path=r"path_to_extracted_driver\chromedriver.exe")

url='https://www.agoda.com/en-gb/search?guid=c342dc8e-11a9-4a8d-b49e-1e0ced85f33c&asq=Ss5PXyh1QUNdFOc4lzIDoPF%2BRvl%2F2EATmGvZScKd0zW3IquUAexOTl%2FhzaLZmWnRrt%2BNsyLXVMPCsbOgn6Txuuyv3GjUsrzxx03ORjNoSvhT2zzzZI16VjZVUfMaJahsG5stZZscvjQurvqGHKlLj8Cw5LYDIbsL%2F6%2BnvlyNDcj4XXyYByZxSESJKBx23i4BmEzrQNpYv5NC385h7l4mGQ%3D%3D&city=1460&tick=638634052144&locale=en-gb&ckuid=feb9150c-a849-46f5-a7d8-a9e0bdf18f0d&prid=0&currency=EUR&correlationId=9c3498c8-b7f3-4925-8216-309ffdae0017&analyticsSessionId=-8262031566606665745&pageTypeId=1&realLanguageId=16&languageId=1&origin=NL&stateCode=NH&cid=-999&userId=feb9150c-a849-46f5-a7d8-a9e0bdf18f0d&whitelabelid=1&loginLvl=0&storefrontId=3&currencyId=1&currencyCode=EUR&htmlLanguage=en-gb&cultureInfoName=en-gb&machineName=am-pc-4g-acm-web-user-bc844b769-94km9&trafficGroupId=4&trafficSubGroupId=6&aid=178961&useFullPageLogin=true&cttp=4&isRealUser=true&mode=production&browserFamily=Chrome&cdnDomain=agoda.net&checkIn=2024-10-10&checkOut=2024-10-12&rooms=1&adults=2&children=0&priceCur=EUR&los=2&textToSearch=Tallinn&travellerType=1&familyMode=off&ds=dwV3vHr93jbaDuoq&productType=-1'
driver.get(url=url)
driver.implicitly_wait(5)

In [ ]:
page_sources=[]
pages=4

try:
    # Wait for the cookie popup and click 'Accept all' or 'Reject all'
    cookie_popup = WebDriverWait(driver, 30).until(
        
        EC.visibility_of_element_located((By.XPATH, "//*[@id='consent-banner-container']/div/div[2]/div/button"))).click() #AGODA


    
except TimeoutException:
    print("Cookie consent popup not found")
    
for i in range(1,pages+1):
  print(f"page {i}")
  for i in range(1,6):
    # Scrolldown
    try:
        pixels=3000
        time.sleep(10) 
        driver.execute_script(f"window.scrollBy(0, {pixels*i})") 
        print(f"Scrolled {pixels*i} pixels")
     
    except:
     print('An exception occurred when scrolling down')
    
    # At end of page, append the html
  page_sources.append(driver.page_source)
    
  # Click next
  try:
    print("Finding next page...")
    next_button = driver.find_element(By.XPATH,'//*[@id="paginationNext"]')
    if next_button:
      print("clicking")
      WebDriverWait(driver,5,
                ignored_exceptions=(
                        NoSuchElementException, 
                        StaleElementReferenceException)).until(EC.visibility_of_element_located((By.XPATH,'//*[@id="paginationNext"]'))).click()
  except Exception as e:
    print(f'An exception occurred when clicking next: {e}')
  

In [5]:
# with open('output.txt', 'w') as f:
#     f.write(str(page_sources))  # Convert the array to a string with square brackets
# 
# print("Array written to file with square brackets.")

In [2]:
import ast

In [3]:
with open('output.txt', 'r') as f:
    page_sources=ast.literal_eval(f.read())

In [ ]:
len(page_sources)

In [9]:
page1=page_sources[0]

In [8]:
for page in page_sources:
    soup = BeautifulSoup(page, 'html.parser')
    hotels_soup=soup.find_all('li',class_='PropertyCard')
    name = rating = address = stars = price = "N/A"
    images = []
    hotels=[]
    number_of_nights=3
    

    for element in hotels_soup:
        try:
            # Initialize variables with default values (None) in case extraction fails
            name = rating = address = text_rating = price = None
            images = []

            # Extract the hotel name (set None if it fails)
            try:
                name = element.find('h3').get_text(strip=True)
            except AttributeError:
                name = None

            # Extract the rating (check if there's at least one <p> tag with that class)
            try:
                p_tags = element.find_all('p', class_='dynamic-style-typographystyle-3')
                if len(p_tags) > 0:
                    rating = p_tags[0].get_text(strip=True)
                else:
                    rating = None
            except AttributeError:
                rating = None

            # # Extract the text rating (check if there's a second <p> tag)
            try:
                stars = element.find('span',class_='sc-crrsfI')
                
                if stars:
                    stars = stars.get_text(strip=True).split(' ')[0]
                else:
                    stars = None
            except AttributeError:
                price("Error here1")
                stars = None

            # Extract the address (set None if it fails)
            try:
                address = element.find('span', class_='dynamic-style-typographystyle-3').get_text(strip=True).split(" - ")[0]
            except AttributeError:
                address = None

            # Extract image URLs (set an empty list if it fails)
            # try:
            #     image_tags = element.find_all('img', class_='thumbnail-image')
            #     images = [img['src'] for img in image_tags if 'src' in img.attrs]
            # except AttributeError:
            #     images = []

            # Extract price per night (set None if it fails)
            try:
                price = element.find('span', class_='PropertyCardPrice__Value').get_text(strip=True)
            except AttributeError:
                price = None

            # Append the extracted data to the hotels list
            hotels.append({
                "name": name,
                "rating": rating,
                "address": address,
                "hotel rating": stars,
                # "images": images,  # Uncomment if you need to include the image URLs
                "price per night($)": price,
            })
        
        except Exception as e:
            print(f"An error occurred while processing a hotel: {e}")
            continue


In [7]:
len(hotels)

24

In [10]:
import pandas as pd

df = pd.DataFrame(hotels)

# Convert the price column to float

df['price per night($)'] = df['price per night($)'].str.replace(',', '').astype(float)

df
df.dropna(subset=['rating'],inplace=True)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 0 to 23
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                12 non-null     object 
 1   rating              12 non-null     object 
 2   address             12 non-null     object 
 3   hotel rating        12 non-null     object 
 4   price per night($)  12 non-null     float64
dtypes: float64(1), object(4)
memory usage: 576.0+ bytes


In [62]:
df["total price ($)"]=df["price per night($)"]*number_of_nights

In [11]:
df

,name,rating,address,hotel rating,price per night($)
0,Old Town - Dunkri Apartment,9.0,"Tallinn Old Town, Tallinn",4,155.0
1,AirHome - Owl's Nest,9.1,"Kalamaja, Tallinn",4.5,323.0
2,3 room central apartmend 90m2 parking for one car,9.4,Managed by a private host,5,76.0
3,Tallinn Central City apartment,7.9,"Lower Town, Tallinn",tooltip,80.0
9,Paivilla Boutique Hotel,8.0,"Kristiine, Tallinn",tooltip,74.0
11,Saia Forest House,9.6,"Suburbs, Tallinn",tooltip,93.0
12,room to the east,9.0,Managed by a private host,tooltip,55.0
13,Rixwell Viru Square Hotel Tallinn,8.0,"Tallinn Old Town, Tallinn",3,66.0
14,Stereo House by Larsen,9.0,"Nomme, Tallinn",tooltip,84.0
20,Railrooms Traincar Hostel,7.9,"Kalamaja, Tallinn",tooltip,59.0


In [ ]:
hotels_soup[0].find_all('span',class_='sc-crrsfI')#.get_text(strip=True)